# Desenvolvendo a Análise e Visualizando os dados

#### Importando bibliotecas:

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import math

#### Carregando os dados:

In [3]:
df_clientes = pd.read_csv(r'./data/dados_clientes.csv')

#### Exibindo informações básicas:

In [4]:
df_clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_cliente         30 non-null     int64  
 1   nome               30 non-null     object 
 2   idade              28 non-null     float64
 3   genero             30 non-null     object 
 4   renda_anual        29 non-null     float64
 5   categoria_produto  30 non-null     object 
 6   valor_compra       28 non-null     float64
 7   data_compra        26 non-null     object 
 8   regiao             30 non-null     object 
dtypes: float64(3), int64(1), object(5)
memory usage: 2.2+ KB


In [5]:
df_clientes.describe()

,id_cliente,idade,renda_anual,valor_compra
count,30.000000,28.000000,29.000000,28.000000
mean,1009.733333,38.892857,157162.103675,3920.147585
std,5.918605,12.561963,142393.899659,4108.469976
min,1000.000000,18.000000,39809.346415,984.100186
25%,1005.250000,31.000000,72155.261524,2017.981504
50%,1010.000000,40.000000,165339.227930,2612.869185
75%,1014.000000,42.250000,174931.504986,4393.001473
max,1019.000000,62.000000,750000.000000,20000.000000


#### Visualizando as 5 primeiras linhas:

In [6]:
df_clientes.head(5)

,id_cliente,nome,idade,genero,renda_anual,categoria_produto,valor_compra,data_compra,regiao
0,1006,Gabriela Rocha,40.0,Masculino,750000.000000,Eletrônicos,1084.904523,2024-01-15,Centro-Oeste
1,1019,Isabela Martins,42.0,Outro,174931.504986,Casa,984.100186,2024-04-05,Norte
2,1014,Gabriela Rocha,33.0,Outro,180660.619728,Moda,2017.981504,2024/03/10,Norte
3,1010,Sabrina Torres,58.0,Feminino,95133.980567,Esporte,4981.455814,2024/03/10,Nordeste
4,1007,Daniel Costa,42.0,Outro,96879.401953,Beleza,2052.315229,2024-06-22,Centro-Oeste


#### Visualizando todo o formato do Data Frame de 'clientes':

In [7]:
df_clientes.shape

(30, 9)

#### Exibindo os tipos de cada coluna:

In [8]:
df_clientes.dtypes

id_cliente             int64
nome                  object
idade                float64
genero                object
renda_anual          float64
categoria_produto     object
valor_compra         float64
data_compra           object
regiao                object
dtype: object

#### Exibindo e Tratando dados ausentes:

In [9]:
# 1) Normalização: tira espaços e padroniza vazios -> NaN
col = (df_clientes['data_compra']
         .astype(str)
         .str.strip()
         .replace({'': np.nan, 'None': np.nan, 'NaN': np.nan, 'nan': np.nan}))

# Normaliza separadores raros, se houver
col = (col
       .str.replace(r'[\._]', '-', regex=True)  # troca '.' e '_' por '-'
       .str.replace(r'\s+', '', regex=True))     # remove espaços internos

# 2) Máscaras de formato (com fillna(False) para evitar NaN em masks)
mask_ddmmyyyy        = col.str.match(r'^\d{2}/\d{2}/\d{4}$').fillna(False)   # 15/02/2024
mask_yyyy_mm_dd      = col.str.match(r'^\d{4}-\d{2}-\d{2}$').fillna(False)   # 2024-01-15
mask_yyyy_mm_dd_slash= col.str.match(r'^\d{4}/\d{2}/\d{2}$').fillna(False)   # 2024/03/10

# 3) Série destino como datetime, alinhada ao índice do df_clientes
parsed = pd.Series(pd.NaT, index=df_clientes.index, dtype='datetime64[ns]')

# 4) Converte cada grupo com o format correto (usando .loc e errors='coerce')
parsed.loc[mask_ddmmyyyy]         = pd.to_datetime(col.loc[mask_ddmmyyyy],         format='%d/%m/%Y', errors='coerce')
parsed.loc[mask_yyyy_mm_dd]       = pd.to_datetime(col.loc[mask_yyyy_mm_dd],       format='%Y-%m-%d', errors='coerce')
parsed.loc[mask_yyyy_mm_dd_slash] = pd.to_datetime(col.loc[mask_yyyy_mm_dd_slash], format='%Y/%m/%d', errors='coerce')

# 5) Atribui ao DataFrame
df_clientes['data_compra'] = parsed

# 6) Formata para dd/mm/yyyy (mantendo NaN onde não foi possível converter)
df_clientes['data_compra'] = df_clientes['data_compra'].dt.strftime('%d-%m-%Y')

C:\Users\A1CLNDO\AppData\Local\Temp\ipykernel_24900\1862909962.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mask_ddmmyyyy        = col.str.match(r'^\d{2}/\d{2}/\d{4}$').fillna(False)   # 15/02/2024
C:\Users\A1CLNDO\AppData\Local\Temp\ipykernel_24900\1862909962.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mask_yyyy_mm_dd      = col.str.match(r'^\d{4}-\d{2}-\d{2}$').fillna(False)   # 2024-01-15
C:\Users\A1CLNDO\AppData\Local\Temp\ipykernel_24900\1862909962.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is d

In [10]:
# Contagem de nulos por coluna
df_clientes.isna().sum()

id_cliente           0
nome                 0
idade                2
genero               0
renda_anual          1
categoria_produto    0
valor_compra         2
data_compra          4
regiao               0
dtype: int64

In [11]:
# Somente colunas que têm ao menos um nulo
cols_com_nulos = df_clientes.columns[df_clientes.isna().any()]
print('Aqui estão todas as colunas com valores ausentes:', cols_com_nulos.tolist())

# Colunas completamente nulas (100% nulos)
cols_100_nulos = df_clientes.columns[df_clientes.isna().all()]
print('Aqui estão todas as linhas com valores ausentes:', cols_100_nulos.tolist())

Aqui estão todas as colunas com valores ausentes: ['idade', 'renda_anual', 'valor_compra', 'data_compra']
Aqui estão todas as linhas com valores ausentes: []


In [12]:
# As datas percorem desde de janeiro de 2024 a junho de 2024...
# Dessa  forma, não é necessário uma acertividade tão grande, somente
# selecionar uma média ou um valor aleatório durante esse período.
# 1. Coluna 'data_compra':
# Converte 'data_compra' (string dd/mm/yyyy) para datetime para facilitar a imputação
dt = pd.to_datetime(df_clientes['data_compra'], format='%d-%m-%Y', errors='coerce')

# Define intervalo
inicio = pd.Timestamp('2024-01-01')
fim    = pd.Timestamp('2024-06-30')

# Cria datas aleatórias para cada NaT
np.random.seed(42)  # opcional, para reprodutibilidade
n_nulos = dt.isna().sum()
dias_total = (fim - inicio).days + 1
datas_random = inicio + pd.to_timedelta(np.random.randint(0, dias_total, size=n_nulos), unit='D')

# Preenche os NaT com as datas aleatórias
dt.loc[dt.isna()] = datas_random

# Volta a formatar para dd/mm/yyyy
df_clientes['data_compra'] = dt.dt.strftime('%d/%m/%Y')

# Garantir que é datetime
df_clientes['data_compra'] = pd.to_datetime(df_clientes['data_compra'])

C:\Users\A1CLNDO\AppData\Local\Temp\ipykernel_24900\2520675520.py:25: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_clientes['data_compra'] = pd.to_datetime(df_clientes['data_compra'])


In [13]:
# 2. Coluna 'idade':
# Exemplo: preencher idade pela média do gênero:
df_clientes['idade'] = df_clientes.groupby('genero')['idade'].transform(lambda x: x.fillna(math.floor(x.mean())))

In [14]:
# 3. Coluna 'valor_compra':
# Preencher valor_compra pela média da categoria:
df_clientes['valor_compra'] = df_clientes.groupby('categoria_produto')['valor_compra'].transform(lambda x: x.fillna(x.mean()))

In [15]:
# 4. Coluna 'renda_anual':
# Preencher renda_anual pela média da categoria:
df_clientes['renda_anual'] = df_clientes.groupby('categoria_produto')['renda_anual'].transform(lambda x: x.fillna(x.mean()))

In [16]:
# Validando se não existem mais valores nulos:
df_clientes.isna().sum()

id_cliente           0
nome                 0
idade                0
genero               0
renda_anual          0
categoria_produto    0
valor_compra         0
data_compra          0
regiao               0
dtype: int64